In [95]:
os.chdir(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

In [96]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [97]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [98]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self):

        config = self.config.data_ingestion

        create_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
    )

        return create_ingestion_config
    


In [99]:
import os
import urllib.request as request
import zipfile 
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [100]:
import os

os.makedirs("artifacts/data_ingestion", exist_ok=True)

In [101]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
        )
            logger.info(f'{filename} download! with the following info: \n{headers}')
        else:
            logger.info(f'File already exists of size: {get_size(Path(self.config.local_data_file))}')



    def extract_zip_file(self):
        '''
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        '''
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
    

In [102]:
try:
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)

    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-06-05 11:44:57,765: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-05 11:44:57,768: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-05 11:44:57,771: INFO: common: created directory at: artifacts]
[2026-06-05 11:45:01,089: INFO: 814952226: artifacts/data_ingestion/data.zip download! with the following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 8E48:105AD0:4D9FA:5F3E2:6A2268B1
Accept-Ranges: bytes
Date: Fri, 05 Jun 2026 06:15:01 GMT
Via: 1.1 varnish
X-Served-By: cache-del-vibw2260031-DEL
X-Cache: HIT
X-Cache-Hits: 0
X-Timer: S1780640102.838822,VS0,VE0
Vary: Authorization,Accep